# Random Forest

In diesem Projekt erkunden wir öffentlich verfügbare Daten von [LendingClub.com](www.lendingclub.com). Lending Club bringt Menschen, die Geld brauchen (Kreditnehmende), mit Menschen zusammen, die Geld haben (Investierende). Wenn du investierst, setzt du hoffentlich auf Menschen, deren Profil darauf hindeutet, dass sie dir das Geld mit hoher Wahrscheinlichkeit zurückzahlen. Wir versuchen, ein Modell zu erstellen, das dabei hilft, das vorherzusagen.

Lending Club hatte ein [sehr interessantes Jahr 2016](https://en.wikipedia.org/wiki/Lending_Club#2016). Sehen wir uns also einige der Daten an und behalten dabei diesen Kontext im Hinterkopf. Die Daten stammen aus der Zeit, bevor das Unternehmen überhaupt an die Börse ging.

Wir verwenden Kreditdaten aus den Jahren 2007 bis 2010 und versuchen zu klassifizieren und vorherzusagen, ob die kreditnehmende Person ihren Kredit vollständig zurückgezahlt hat oder nicht. Du kannst die Daten [hier](https://www.lendingclub.com/info/download-data.action) herunterladen oder einfach die bereitgestellte CSV-Datei verwenden. Am besten verwendest du die bereitgestellte CSV-Datei, denn sie ist bereits von NA-Werten bereinigt.

Das bedeuten die Spalten:
* credit.policy: 1, wenn die Person die Kriterien der Kreditprüfung von LendingClub.com erfüllt, sonst 0.
* purpose: Der Verwendungszweck des Kredits (nimmt die Werte "credit_card", "debt_consolidation", "educational", "major_purchase", "small_business" und "all_other" an).
* int.rate: Der Zinssatz des Kredits als Anteil (ein Zinssatz von 11% würde als 0,11 gespeichert). Kreditnehmende, die LendingClub.com als riskanter einstuft, bekommen höhere Zinssätze.
* installment: Die monatlichen Raten, die die kreditnehmende Person schuldet, wenn der Kredit bewilligt wird.
* log.annual.inc: Der natürliche Logarithmus des selbst angegebenen Jahreseinkommens der kreditnehmenden Person.
* dti: Das Verhältnis von Schulden zu Einkommen der kreditnehmenden Person (Schuldenbetrag geteilt durch das Jahreseinkommen).
* fico: Der FICO-Score der kreditnehmenden Person.
* days.with.cr.line: Die Anzahl der Tage, die die kreditnehmende Person schon eine Kreditlinie hat.
* revol.bal: Der revolvierende Saldo der kreditnehmenden Person (der am Ende des Abrechnungszeitraums der Kreditkarte unbezahlte Betrag).
* revol.util: Die Auslastungsquote der revolvierenden Kreditlinie der kreditnehmenden Person (der genutzte Betrag der Kreditlinie im Verhältnis zum insgesamt verfügbaren Kredit).
* inq.last.6mths: Die Anzahl der Anfragen von Kreditgebenden zur kreditnehmenden Person in den letzten 6 Monaten.
* delinq.2yrs: Wie oft die kreditnehmende Person in den letzten 2 Jahren mit einer Zahlung mehr als 30 Tage im Zahlungsverzug war.
* pub.rec: Die Anzahl der negativen öffentlichen Einträge der kreditnehmenden Person (Insolvenzanträge, Steuerpfandrechte oder Gerichtsurteile).
* not.fully.paid: 1, wenn der Kredit nicht vollständig zurückgezahlt wurde, sonst 0.

## Lernziele

Am Ende dieses Notebooks kannst du:

- die Lending-Club-Daten mit Histogrammen, einem Countplot und facettierten Regressionsplots erkunden und interpretieren, was jeder dieser Plots zeigt.
- das kategoriale Feature `purpose` mit `pd.get_dummies` kodieren und die Daten in einen Trainings- und einen Testdatensatz aufteilen.
- einen Entscheidungsbaum und einen Random Forest trainieren und vergleichen, um Kreditausfälle vorherzusagen.
- beide Modelle mit einem Classification Report und einer Confusion Matrix evaluieren, dem Klassenungleichgewicht mit `class_weight` entgegenwirken und die Feature Importances in eine Rangfolge bringen.

## Bibliotheken importieren

**Importiere die üblichen Bibliotheken für pandas und die Darstellung. sklearn kannst du später importieren.**

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["patch.force_edgecolor"] = True

## Daten beschaffen

**Lies loan_data.csv mit pandas als DataFrame namens loans ein.**

In [ ]:
loans = pd.read_csv('../data/loan_data.csv')

**Sieh dir die Methoden info(), head() und describe() für loans an.**

In [ ]:
print(loans.info())

In [ ]:
loans.describe()

In [ ]:
loans.head()

## Explorative Datenanalyse

Jetzt visualisieren wir die Daten! Wir verwenden seaborn und die eingebauten Plotfunktionen von pandas, aber du kannst gern jede Bibliothek nutzen, die du möchtest. Mach dir keine Gedanken, ob die Farben übereinstimmen. Wichtig ist nur, dass du die Kernaussage des Plots hinbekommst.

**Erstell ein Histogramm aus zwei übereinanderliegenden FICO-Verteilungen, eine für jede Ausprägung von credit.policy.**

*Hinweis: Das ist ziemlich knifflig, schau ruhig in die Lösungen. Wahrscheinlich brauchst du eine Codezeile pro Histogramm. Ich empfehle außerdem, einfach das in pandas eingebaute .hist() zu verwenden.*

In [ ]:
plt.figure(figsize=(10,5))
loans[loans['credit.policy']==1]['fico'].hist(bins=30, color='blue', label='Policy: 0', alpha=0.6)
loans[loans['credit.policy']==0]['fico'].hist(bins=30, color='red', label='Policy: 1', alpha=0.6)
plt.legend()

**Erstell eine ähnliche Abbildung, aber wähl die Daten diesmal nach der Spalte not.fully.paid aus.**

In [ ]:
plt.figure(figsize=(10,6))
loans[loans['not.fully.paid']==0]['fico'].hist(bins=30, color='blue', label='Fully Paid', alpha=0.4)
loans[loans['not.fully.paid']==1]['fico'].hist(bins=30, color='red', label='Not fully Paid', alpha=0.5)
plt.legend()

**Erstell mit seaborn einen Countplot, der die Anzahl der Kredite nach Verwendungszweck zeigt, eingefärbt nach not.fully.paid.**

In [ ]:
plt.figure(figsize = (11,6))
sns.countplot(x=loans['purpose'], hue=loans['not.fully.paid'], palette = 'Set1')

<details>
<summary>Antwort anzeigen</summary>

Die Kredite für `debt_consolidation` und `credit_card` dominieren die reinen Anzahlen, daher stellen sie auch die größte absolute Zahl nicht vollständig zurückgezahlter Kredite. Volumen ist aber nicht dasselbe wie Risiko: Bei Krediten für `small_business` ist der *Anteil* nicht vollständig zurückgezahlter Kredite im Verhältnis zu ihrer Gesamtzahl viel höher, daher wirken sie unverhältnismäßig riskant. Diese Lesart ist nützlicher, als nur festzustellen, dass es viele Kredite zur Umschuldung gibt.
</details>

**Sehen wir uns den Trend zwischen FICO-Score und Zinssatz an. Bau den folgenden Jointplot nach.**

In [ ]:
sns.jointplot(x='fico', y='int.rate', data=loans, kind='hex')

**Der `jointplot` oben hat den Gesamttrend FICO vs Zinssatz gezeigt. Stellen wir jetzt eine gezieltere Frage: Unterscheidet sich dieser Trend je nachdem, *warum* Menschen einen Kredit aufnehmen?**

**Stell mit `sns.lmplot()` `int.rate` gegen `fico` dar und facettiere dabei in eine Spalte pro `purpose` des Kredits (verwende `col='purpose'` mit `col_wrap=3`). Übergib `scatter_kws={'alpha': 0.3}`, damit überlappende Punkte lesbar bleiben.**

Das ist aussagekräftiger als eine Aufteilung nach `credit.policy`, denn `credit.policy` ist selbst aus FICO abgeleitet. Eine Aufteilung nach FICO zeichnet dort also meist nur dieselbe Linie noch einmal. Der Verwendungszweck des Kredits ist unabhängig vom Score, daher können sich die Regressionsgeraden pro Verwendungszweck tatsächlich unterscheiden.

> Zur Erinnerung: `lmplot()` ist ein Streudiagramm mit überlagerter Regressionsgerade (`regplot()`), eingebettet in ein `FacetGrid()`, damit du den Zusammenhang nach Teilmengen aufschlüsseln kannst. Sieh dir die [Dokumentation zu lmplot()](https://seaborn.pydata.org/generated/seaborn.lmplot.html) und die [Erklärung zu regplot vs lmplot](https://seaborn.pydata.org/tutorial/regression.html) an.

In [ ]:
sns.lmplot(x='fico', y='int.rate', data=loans, col='purpose', col_wrap=3,
           height=3, scatter_kws={'alpha': 0.3})

<details>
<summary>Antwort anzeigen</summary>

Bei jedem Verwendungszweck ist der Zusammenhang negativ: Höhere FICO-Scores gehen mit niedrigeren Zinssätzen einher, was wir auch erwarten würden. Interessant ist, dass die Steigung nicht überall gleich ist. Bei `small_business` und `debt_consolidation` hängen die Zinsen stärker vom Kreditscore ab, während dünner besetzte Kategorien wie `educational` flacher und verrauschter sind (teilweise, weil es dort viel weniger Kredite gibt und das Regressionsband deshalb breit ist). Das ist ehrlicher als der einzelne `jointplot`: Der Hexplot über alle Daten hat verborgen, dass die Höhe der Zinsen davon abhängt, *warum* jemand einen Kredit aufnimmt, nicht nur vom Score. Eine Aufteilung nach `credit.policy` würde das dagegen nicht zeigen, weil `credit.policy` selbst eine Regel auf Basis von FICO ist. Beide Gruppen würden also im Wesentlichen dieselbe Linie nachzeichnen.
</details>

## Daten aufbereiten

Bereiten wir unsere Daten für unser Random-Forest-Klassifikationsmodell vor!

**Sieh dir loans.info() noch einmal an.**

In [ ]:
loans.info()

### Kategoriale Features

Beachte, dass die Spalte **purpose** kategorial ist

Das heißt, wir müssen sie mit Dummy-Variablen umwandeln, damit sklearn sie verstehen kann. Das erledigen wir in einem sauberen Schritt mit pd.get_dummies.

Wir zeigen dir einen Weg, mit diesen Spalten umzugehen, der sich bei Bedarf auf mehrere kategoriale Features erweitern lässt.

**Erstell eine Liste mit 1 Element, die den String 'purpose' enthält. Nenn diese Liste cat_feats.**

In [ ]:
cat_feats = ['purpose']

In [ ]:
cat_feats

**Verwende jetzt pd.get_dummies(loans,columns=cat_feats,drop_first=True), um einen aufbereiteten, größeren DataFrame mit neuen Feature-Spalten aus Dummy-Variablen zu erstellen. Speichere diesen DataFrame als final_data.**

In [ ]:
final_data = pd.get_dummies(loans, columns=cat_feats, drop_first=True)

In [ ]:
final_data.head()

### Aufteilung in Trainings- und Testdaten

Jetzt ist es Zeit, unsere Daten in einen Trainings- und einen Testdatensatz aufzuteilen!

**Teil deine Daten mit sklearn in einen Trainings- und einen Testdatensatz auf, wie wir es schon früher gemacht haben.**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = final_data.drop('not.fully.paid', axis=1)
y = final_data['not.fully.paid']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

## Entscheidungsbaummodell trainieren

Zuerst trainieren wir einen einzelnen Entscheidungsbaum!

**Importiere DecisionTreeClassifier**

In [ ]:
from sklearn.tree import DecisionTreeClassifier

**Erstell eine Instanz von DecisionTreeClassifier() namens dtree und trainiere sie auf den Trainingsdaten.**

In [ ]:
dtree = DecisionTreeClassifier()

In [ ]:
dtree.fit(X_train, y_train)

### Vorhersagen und Evaluation des Entscheidungsbaums
**Erstell Vorhersagen für den Testdatensatz, dazu einen Classification Report und eine Confusion Matrix.**

In [ ]:
predictions = dtree.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
print(classification_report(y_test, predictions))

In [ ]:
print(confusion_matrix(y_test, predictions))

## Random-Forest-Modell trainieren

Jetzt ist es Zeit, unser Modell zu trainieren!

**Erstell eine Instanz der Klasse RandomForestClassifier und trainiere sie auf unseren Trainingsdaten aus dem vorherigen Schritt.**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rnc = RandomForestClassifier(n_estimators = 200)

In [ ]:
rnc.fit(X_train, y_train)

### Vorhersagen und Evaluation

Machen wir Vorhersagen anhand der Werte von y_test und evaluieren wir unser Modell.

**Sag die Klasse von not.fully.paid für die Daten in X_test vorher.**

In [ ]:
pred = rnc.predict(X_test)

**Erstell jetzt aus den Ergebnissen einen Classification Report. Fällt dir etwas Seltsames auf, oder bekommst du eine Warnung?**

In [ ]:
print(classification_report(y_test, pred))

**Zeig die Confusion Matrix für die Vorhersagen an.**

In [ ]:
print(confusion_matrix(y_test, pred))

## Deine Ergebnisse interpretieren

**Der Random Forest hat die höhere Gesamt-Accuracy, aber was ist mit seinem Recall für Klasse 1 passiert (Kredite, die nicht vollständig zurückgezahlt wurden)? Welches Modell ist für Investierende tatsächlich nützlicher, und warum ist die Accuracy hier irreführend?**

<details>
<summary>Antwort anzeigen</summary>

Der einzelne Entscheidungsbaum hat etwa ein Viertel der riskanten Kredite erkannt (Recall um 0,24 für Klasse 1), auf Kosten von mehr Fehlalarmen. Der Random Forest hat eine höhere Gesamt-Accuracy erzielt (etwa 0,84 vs 0,73), aber sein Recall für Klasse 1 ist auf ungefähr 0,02 eingebrochen: Er sagt für fast alle „vollständig zurückgezahlt“ vorher. Da nur 16% der Kredite ausfallen, erreicht schon die ständige Vorhersage „vollständig zurückgezahlt“ etwa 84% Accuracy. Die Accuracy belohnt also genau das falsche Verhalten. Investierenden kommt es darauf an, die faulen Kredite zu erkennen, also auf den Recall für Klasse 1. Deshalb ist der Random Forest mit der hohen Accuracy in der trainierten Form praktisch kaum von Nutzen. Genau dieses Problem behebt der nächste Schritt.
</details>

### Umgang mit dem Klassenungleichgewicht

Nur etwa 16% der Kredite in diesem Datensatz wurden nicht vollständig zurückgezahlt. Ein Modell kann also eine hohe Accuracy erreichen, indem es fast immer „vollständig zurückgezahlt“ vorhersagt. Eine einfache Lösung besteht darin, dem Modell mit `class_weight='balanced'` zu sagen, dass es die seltene Klasse stärker gewichten soll.

**Trainiere den Random Forest mit `class_weight='balanced'` neu (behalte `n_estimators=200` bei und setz `random_state=101` für die Reproduzierbarkeit), mach Vorhersagen für `X_test` und vergleich den Classification Report und die Confusion Matrix mit der unbalancierten Version.**

In [ ]:
rnc_balanced = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=101)
rnc_balanced.fit(X_train, y_train)
pred_balanced = rnc_balanced.predict(X_test)

print(classification_report(y_test, pred_balanced))
print(confusion_matrix(y_test, pred_balanced))

<details>
<summary>Antwort anzeigen</summary>

Durch das Balancieren der Klassengewichte achtet das Modell stärker auf die seltene Klasse. Der Recall für Klasse 1 steigt von etwa 0,02 auf ungefähr 0,12: immer noch niedrig, aber wir markieren jetzt rund 55 der tatsächlichen Ausfälle statt 9. Der Preis ist der erwartete Trade-off: Die Precision für Klasse 1 bleibt schwach (etwa 0,29), und die Gesamt-Accuracy sinkt leicht (von etwa 0,84 auf 0,82), weil wir mehr Fehlalarme bei Krediten auslösen, die tatsächlich zurückgezahlt wurden. `class_weight='balanced'` ist eine moderate Anpassung, keine vollständige Lösung: Um den Recall deutlich zu steigern, bräuchte es Resampling, das Tuning des Schwellenwerts oder bessere Features. Die eigentliche Lehre ist, dass eine einzelne Accuracy-Zahl all das verdeckt hat.
</details>

### Welche Features bestimmen die Vorhersage?

Ein Random Forest zeigt über `.feature_importances_`, wie viel jedes Feature zu seinen Aufteilungen beigetragen hat. Wenn wir sie darstellen, wird aus einem undurchsichtigen Modell etwas, das wir mit unserer EDA abgleichen können.

**Bau eine sortierte pandas Series aus den Feature Importances des Random Forests und stell sie als horizontales Balkendiagramm dar.**

In [ ]:
importances = pd.Series(rnc.feature_importances_, index=X.columns).sort_values()

plt.figure(figsize=(10, 6))
importances.plot(kind='barh')
plt.title('Random Forest Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()

<details>
<summary>Antwort anzeigen</summary>

Zwei Dinge fallen auf. Erstens sammeln sich die stetigen numerischen Spalten (`installment`, `revol.util`, `days.with.cr.line`, `revol.bal`, `dti`, `log.annual.inc`) ganz oben, während die per One-Hot Encoding kodierten Dummy-Variablen `purpose_*` und seltene Indikatorspalten wie `pub.rec` und `credit.policy` weit unten liegen. Zweitens, und das ist überraschender: `fico` und `int.rate`, die Variablen, die in unserer EDA am trennschärfsten wirkten, landen nur im Mittelfeld (`int.rate` etwa auf Platz 7, `fico` etwa auf Platz 8).

Genau diese Abweichung ist die eigentliche Lehre. Die Standard-Feature-Importance (Gini) eines Random Forests ist zugunsten von Features mit vielen verschiedenen Werten verzerrt. Stetige Spalten mit hoher Kardinalität werden dadurch überbewertet, und ein gröberes Feature mit echter Vorhersagekraft wie `fico` kann unterbewertet werden. Betrachte dieses Diagramm als Hinweis, nicht als letztes Wort: Die Permutation Importance oder SHAP-Werte würden eine fairere Rangfolge liefern. „Das Modell sagt, X ist wichtig“ verdient immer einen zweiten Blick.
</details>

## Das war's!